# P105 — SeeClick: aprovechar el anclaje visual para agentes avanzados de interfaz gráfica

## 1. Título y paper

**Paper:** *SeeClick: Harnessing GUI Grounding for Advanced Visual GUI Agents*  
**Autoría:** Kanzhi Cheng, Qiushi Sun, Yougang Chu, Fangzhi Xu, Yantao Li, y otros  
**Año y venue:** 2024 · ACL 2024 · arXiv:2401.10935  
**Nivel:** L2 · **Motor:** `seeclick`  
**Ficha completa:** [`P105_seeclick`](../../papers/foundational/P105_seeclick/README.md)

**Hito:** Aísla el anclaje —de una instrucción a unas coordenadas— como la capacidad que separa describir una pantalla de poder operarla.

- [arXiv:2401.10935](https://arxiv.org/abs/2401.10935)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los agentes de interfaz dependían del árbol de accesibilidad o del HTML: texto estructurado que muchas aplicaciones no exponen, y que no cubre los elementos que solo son un icono. Sin ese texto, el agente no puede ni referirse al botón.
2. Ejecutar una implementación mínima de la propuesta: Trabajar directamente sobre la captura de pantalla y entrenar específicamente el anclaje: dada una instrucción en lenguaje natural, devolver las coordenadas del elemento. Con un banco de pruebas propio para medir esa capacidad por separado.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P18
- P104


## 4. Intuición

Un agente puede describir perfectamente una captura de pantalla y ser incapaz de usarla. Para actuar hace falta convertir «abre los ajustes» en un par de coordenadas, y eso es una capacidad distinta —el anclaje— que se puede medir por separado.


## 5. Concepto mínimo

```text
Árbol de accesibilidad / HTML  →  solo ve lo que tiene texto
Anclaje visual                 →  localiza cualquier elemento por su apariencia

    instrucción en lenguaje natural  ⟶  (x, y) donde pulsar

Los iconos sin etiqueta son invisibles para el primero.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('seeclick', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos elementos de la interfaz no tienen etiqueta de texto?
2. ¿Cuántas instrucciones acierta un agente que solo lee texto?
3. ¿Y uno con anclaje visual?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('seeclick', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('seeclick', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

**3 de 6** elementos son solo icono. El agente que solo lee el árbol de accesibilidad acierta **2 de 5** instrucciones; el que ancla visualmente, **5 de 5**. La diferencia no está en el razonamiento: está en si puede **señalar** el elemento.


## 10. Comentario pedagógico

Separar el anclaje del resto es lo más útil del artículo desde el punto de vista de ingeniería. Permite saber si un agente falla porque no entiende la tarea o porque no encuentra el botón, y esas dos cosas se arreglan de formas completamente distintas.


## 11. Error o anti-patrón deliberado

Anti-patrón: confundir anclaje con planificación.


In [ ]:
print('Saber DONDE pulsar no dice QUE pulsar.')
print('Un agente con anclaje perfecto y mal plan se equivoca con puntería impecable.')
print('Son dos capacidades y conviene medirlas por separado para saber que arreglar.')

## 12. Corrección

Las dos capacidades, medidas aparte:


In [ ]:
r = run_paper_lab('seeclick', seed=7)['result']
print('elementos sin texto  :', r['elementos_solo_con_icono'])
print('solo texto           :', r['agente_solo_texto']['aciertos'])
print('con anclaje visual   :', r['agente_con_anclaje_visual']['aciertos'])
for d in r['agente_solo_texto']['detalle']:
    print('   ', d)

## 13. Desafío guiado

Identifica qué instrucciones falla el agente de solo texto y comprueba que todas apuntan a elementos sin etiqueta.


In [ ]:
r = run_paper_lab('seeclick', seed=3)['result']
show(r)

## 14. Desafío autónomo

Haz una captura de una aplicación que uses, lista sus elementos accionables y marca cuáles tienen etiqueta de texto accesible. Calcula qué proporción quedaría fuera del alcance de un agente sin anclaje visual.


## 15. Evidencia de aprendizaje

Guarda el conteo de elementos sin texto y la comparación de aciertos, con tu distinción entre anclaje y planificación.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P105_seeclick/README.md) · evaluación formal: [`assessments/papers/P105_seeclick.md`](../../assessments/papers/P105_seeclick.md)


## 16. Cierre

Navegador y anclaje resueltos por separado. El escritorio completo —varias aplicaciones a la vez— es donde se ve lo que falta.


## 17. Conexión con el siguiente hito

- P106

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
